In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("Titanic.csv")

df["age"] = df["age"].fillna(df["age"].median())
df["sex"] = (df["sex"] == "male").astype(int)
df = df.drop("who",axis=1)
df = pd.get_dummies(df,columns=["embarked","class","alone"]).astype(int)

In [3]:
df

,sex,age,sibsp,parch,fare,survived,embarked_C,embarked_Q,embarked_S,class_First,class_Second,class_Third,alone_False,alone_True
0,1,22,1,0,7,0,0,0,1,0,0,1,1,0
1,0,38,1,0,71,1,1,0,0,1,0,0,1,0
2,0,26,0,0,7,1,0,0,1,0,0,1,0,1
3,0,35,1,0,53,1,0,0,1,1,0,0,1,0
4,1,35,0,0,8,0,0,0,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,1,27,0,0,13,0,0,0,1,0,1,0,0,1
887,0,19,0,0,30,1,0,0,1,1,0,0,0,1
888,0,28,1,2,23,0,0,0,1,0,0,1,1,0
889,1,26,0,0,30,1,1,0,0,1,0,0,0,1


In [4]:
X = df.drop("survived",axis=1)
y = df["survived"]

In [5]:
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
y_train_scaled = scaler.transform(X_val)

In [7]:
model = keras.Sequential([
    keras.layers.Dense(64,activation="relu",input_shape=(X_train.shape[1],)),
    # keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    # keras.layers.Dense(64,activation="relu"),
    # keras.layers.Dropout(0.3),
    keras.layers.Dense(32,activation="relu"),
    # keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1,activation="sigmoid")
])

c:\Users\Kanhaiya\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

checkpoint = keras.callbacks.ModelCheckpoint(
    filepath="Titanic_model.keras",
    monitor="val_loss",
    save_best_only=True
)

history = model.fit(
    X_train,y_train,
    epochs=1000,
    validation_data=(X_val,y_val),
    callbacks=[early_stop,checkpoint]
)

import pickle
with open('titanic_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"\nVal Loss:     {val_loss:.4f}")
print(f"Val Accuracy: {val_acc:.4f}")


TypeError: ModelCheckpoint.__init__() missing 1 required positional argument: 'filepath'

In [10]:
# # Plot losses
# plt.figure(figsize=(10, 4))

# plt.subplot(1, 2, 1)
# plt.plot(history.history['loss'], label='train')
# plt.plot(history.history['val_loss'], label='validation')
# plt.title('Loss')
# plt.legend()

# plt.subplot(1, 2, 2)
# plt.plot(history.history['accuracy'], label='train')
# plt.plot(history.history['val_accuracy'], label='validation')
# plt.title('Accuracy')
# plt.legend()

# plt.tight_layout()
# plt.show()

# # Final evaluation
# val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
# print(f"\nVal Loss:     {val_loss:.4f}")
# print(f"Val Accuracy: {val_acc:.4f}")